# 10 DINO Patch Matching Probe

역할: mask, prototype, OT를 모두 빼고 DINO patch feature만으로 clean image와 shifted image가 patch level에서 어떻게 대응되는지 확인한다.

확인할 것:
- 같은 위치 patch의 cosine similarity가 shift에서 얼마나 떨어지는가
- query patch의 nearest clean patch가 같은 위치인지, 주변 위치인지, 멀리 이동하는지
- `delta_i = f_shift_i - f_clean_i`가 representation space에서 비슷한 방향/low-dimensional subspace로 움직이는지
- brightness / blur / noise / position shift별 displacement와 feature-shift 패턴


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM` checkout을 사용하고, 로컬에서는 현재 repo root를 찾는다. 이 노트북은 raw LOCO image와 DINO만 사용하며 SAM/GroundingDINO mask를 읽지 않는다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'


def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )


colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(
            [
                'git', '-C', str(colab_checkout), 'checkout', 'FETCH_HEAD', '--',
                'experiments/validation/condition_shift_baseline/src',
                'experiments/validation/condition_shift_baseline/notebook/10_dino_patch_matching_probe.ipynb',
            ],
            check=True,
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((candidate.resolve() for candidate in repo_candidates if candidate.exists() and is_regram_root(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for import_root in [SRC_ROOT, SRC_ROOT / 'orchestration']:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Dataset And Shift Setup

`test/good` clean image 몇 장을 기준으로 high severity shift image를 만든다. 이미 06/09에서 생성된 shifted image가 있으면 재사용하고, 없으면 augmentation만 적용해서 저장한다.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, UnidentifiedImageError

from data.augmentation_runtime import DEFAULT_PARAMS, apply_augmentation

CATEGORY = 'breakfast_box'
SPLITS = ['train', 'validation', 'test']
SAMPLE_IDS = 'all'  # 'all' or a list like ['000.png', '001.png'] applied per split
MAX_SAMPLE_IMAGES_PER_SPLIT = None  # set an int for quick smoke tests
AUGMENTATION_WORKERS = min(8, os.cpu_count() or 1)
SHIFT_CONDITIONS = [
    ('clean_identity', 'identity', 'none'),
    ('brightness_high', 'brightness', 'high'),
    ('gaussian_blur_high', 'gaussian_blur', 'high'),
    ('gaussian_noise_high', 'gaussian_noise', 'high'),
    ('position_shift_high', 'position_shift', 'high'),
]
FEATURE_BACKBONE = 'dinov2_vits14'
IMAGE_SIZE = 224

RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
PROBE_ROOT = EXP_ROOT / 'reports' / 'dino_patch_matching_probe'
SHIFTED_IMAGE_ROOT = PROBE_ROOT / 'images'
SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_dino_patch_match_summary.csv'
PROBE_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')


def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')


def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [parent / 'content' / dataset_name, parent / 'data' / 'row' / dataset_name]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct


def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    if any((RAW_LOCO_ROOT / CATEGORY / split / 'good').exists() for split in SPLITS):
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')


def clean_image_path(split: str, image_id: str) -> Path:
    return RAW_LOCO_ROOT / CATEGORY / split / 'good' / image_id


def shifted_image_path(split: str, image_id: str, condition: str) -> Path:
    return SHIFTED_IMAGE_ROOT / CATEGORY / split / condition / image_id


def stable_seed(split: str, image_id: str, condition: str) -> int:
    return 20260520 + sum(ord(ch) for ch in f'{CATEGORY}/{split}/{image_id}/{condition}')


def image_is_readable(path: Path) -> bool:
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            image.convert('RGB').load()
        return True
    except (OSError, UnidentifiedImageError, SyntaxError):
        return False


def load_rgb_checked(path: Path) -> Image.Image:
    try:
        image = Image.open(path).convert('RGB')
        image.load()
        return image
    except (OSError, UnidentifiedImageError, SyntaxError) as exc:
        raise OSError(f'Failed to read image: {path}') from exc


restore_raw_loco_from_drive_if_needed()
sample_refs = []
for split in SPLITS:
    split_root = RAW_LOCO_ROOT / CATEGORY / split / 'good'
    if not split_root.exists():
        print(f'skip missing split: {split_root}')
        continue
    if SAMPLE_IDS == 'all':
        split_sample_ids = sorted(path.name for path in split_root.glob('*.png'))
    else:
        split_sample_ids = list(SAMPLE_IDS)
    if MAX_SAMPLE_IMAGES_PER_SPLIT is not None:
        split_sample_ids = split_sample_ids[:int(MAX_SAMPLE_IMAGES_PER_SPLIT)]
    sample_refs.extend((split, image_id) for image_id in split_sample_ids)
if not sample_refs:
    raise FileNotFoundError(f'No clean images found for splits={SPLITS} under {RAW_LOCO_ROOT / CATEGORY}')
print(f'num clean images = {len(sample_refs)} across splits={SPLITS}')

missing = [path for split, image_id in sample_refs if not (path := clean_image_path(split, image_id)).exists()]
if missing:
    raise FileNotFoundError(f'Missing clean images: {missing}')

condition_order = {condition: index for index, (condition, _, _) in enumerate(SHIFT_CONDITIONS)}
augmentation_tasks = [
    (split, image_id, condition, augmentation_type, severity)
    for split, image_id in sample_refs
    for condition, augmentation_type, severity in SHIFT_CONDITIONS
]


def build_query_row(task: tuple[str, str, str, str, str]) -> dict:
    split, image_id, condition, augmentation_type, severity = task
    source_path = clean_image_path(split, image_id)
    if condition == 'clean_identity':
        query_path = source_path
        if not image_is_readable(query_path):
            raise OSError(f'Clean source image is not readable: {query_path}')
    else:
        query_path = shifted_image_path(split, image_id, condition)
        if query_path.exists() and not image_is_readable(query_path):
            print(f'regenerate truncated shifted image: {query_path}')
            query_path.unlink()
        if not query_path.exists():
            query_path.parent.mkdir(parents=True, exist_ok=True)
            source_image = load_rgb_checked(source_path)
            shifted = apply_augmentation(
                source_image,
                augmentation_type=augmentation_type,
                severity=severity,
                seed=stable_seed(split, image_id, condition),
                params=DEFAULT_PARAMS[augmentation_type][severity],
            )
            shifted.save(query_path)
            if not image_is_readable(query_path):
                query_path.unlink(missing_ok=True)
                raise OSError(f'Generated shifted image is not readable: {query_path}')
    return {
        'split': split,
        'image_id': image_id,
        'condition': condition,
        'augmentation_type': augmentation_type,
        'severity': severity,
        'reference_path': str(source_path),
        'query_path': str(query_path),
        'condition_order': condition_order[condition],
    }


print(f'prepare shifted images with workers={AUGMENTATION_WORKERS}')
if AUGMENTATION_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=AUGMENTATION_WORKERS) as pool:
        query_rows = list(pool.map(build_query_row, augmentation_tasks))
else:
    query_rows = [build_query_row(task) for task in augmentation_tasks]

query_df = (
    pd.DataFrame(query_rows)
    .sort_values(['split', 'image_id', 'condition_order'])
    .drop(columns=['condition_order'])
    .reset_index(drop=True)
)
display(query_df)


## Cell Role: DINO Setup

DINOv2 patch feature extractor를 준비한다. 이후 모든 비교는 DINO patch feature만 사용한다.


In [ ]:
import numpy as np
import torch

from stage1_adapter import (
    analyze_dino_representation_shift,
    match_dino_patch_grid,
    summarize_dino_patch_match,
    summarize_dino_representation_shift,
)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

dino_model = torch.hub.load('facebookresearch/dinov2', FEATURE_BACKBONE).to(device).eval()
dino_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32, device=device).view(1, 3, 1, 1)
dino_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32, device=device).view(1, 3, 1, 1)


DINO_BATCH_SIZE = 32 if torch.cuda.is_available() else 8
USE_DINO_AMP = bool(torch.cuda.is_available())


def preprocess_dino(image: Image.Image) -> torch.Tensor:
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    return tensor


def preprocess_dino_batch(image_paths: list[Path]) -> torch.Tensor:
    tensors = [preprocess_dino(load_rgb_checked(path)) for path in image_paths]
    batch = torch.stack(tensors, dim=0).to(device, non_blocking=True)
    return (batch - dino_mean) / dino_std


@torch.no_grad()
def extract_dino_feature_maps_batch(image_paths: list[Path], *, batch_size: int = DINO_BATCH_SIZE) -> dict[str, np.ndarray]:
    outputs = {}
    unique_paths = list(dict.fromkeys(Path(path) for path in image_paths))
    for start in range(0, len(unique_paths), batch_size):
        batch_paths = unique_paths[start:start + batch_size]
        batch = preprocess_dino_batch(batch_paths)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_DINO_AMP):
            output = dino_model.forward_features(batch)
        tokens = output['x_norm_patchtokens'] if isinstance(output, dict) else output
        tokens = tokens.detach().cpu().float()
        grid = int(math.sqrt(tokens.shape[1]))
        if grid * grid != tokens.shape[1]:
            raise ValueError(f'DINO token count is not square: {tokens.shape[1]}')
        feature_maps = tokens.reshape(tokens.shape[0], grid, grid, tokens.shape[-1]).numpy()
        for path, feature_map in zip(batch_paths, feature_maps, strict=True):
            outputs[str(path)] = feature_map
        print(f'DINO batch {min(start + batch_size, len(unique_paths))}/{len(unique_paths)}')
    return outputs


@torch.no_grad()
def extract_dino_feature_map(image_path: Path) -> np.ndarray:
    return extract_dino_feature_maps_batch([image_path], batch_size=1)[str(Path(image_path))]

print('DINO backbone ready:', FEATURE_BACKBONE, 'batch_size=', DINO_BATCH_SIZE, 'amp=', USE_DINO_AMP)


## Cell Role: Run DINO-Only Patch Matching

각 query image의 DINO patch를 같은 image의 clean DINO patch와 top-1 nearest-neighbor matching한다. 저장되는 metric은 같은 좌표 similarity, top-1 similarity, top-1이 같은 위치인지, top-1 displacement가 얼마나 큰지다.


In [ ]:
match_results = {}
summary_rows = []

unique_feature_paths = sorted({Path(path) for path in query_df['reference_path'].tolist() + query_df['query_path'].tolist()}, key=str)
print(f'extract DINO features for unique images: {len(unique_feature_paths)}')
feature_cache = extract_dino_feature_maps_batch(unique_feature_paths, batch_size=DINO_BATCH_SIZE)


def cached_feature_map(path: Path) -> np.ndarray:
    key = str(Path(path))
    if key not in feature_cache:
        feature_cache.update(extract_dino_feature_maps_batch([Path(path)], batch_size=1))
    return feature_cache[key]


for _, row in query_df.iterrows():
    reference_path = Path(row['reference_path'])
    query_path = Path(row['query_path'])
    reference_feature_map = cached_feature_map(reference_path)
    query_feature_map = cached_feature_map(query_path)
    result = match_dino_patch_grid(reference_feature_map, query_feature_map)
    key = (row['split'], row['image_id'], row['condition'])
    match_results[key] = result
    summary = summarize_dino_patch_match(result)
    summary.update(row.to_dict())
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f'saved summary: {SUMMARY_CSV}')
display(summary_df)

split_condition_summary_df = summary_df.groupby(['split', 'condition']).agg(
    n=('image_id', 'count'),
    identity_similarity_mean=('identity_similarity_mean', 'mean'),
    top1_similarity_mean=('top1_similarity_mean', 'mean'),
    identity_top1_ratio=('identity_top1_ratio', 'mean'),
    local_3x3_top1_ratio=('local_3x3_top1_ratio', 'mean'),
    mean_displacement_patches=('mean_displacement_patches', 'mean'),
    p90_displacement_patches=('p90_displacement_patches', 'mean'),
).reset_index()
display(split_condition_summary_df)

condition_summary_df = summary_df.groupby('condition').agg(
    n=('image_id', 'count'),
    identity_similarity_mean=('identity_similarity_mean', 'mean'),
    top1_similarity_mean=('top1_similarity_mean', 'mean'),
    identity_top1_ratio=('identity_top1_ratio', 'mean'),
    local_3x3_top1_ratio=('local_3x3_top1_ratio', 'mean'),
    mean_displacement_patches=('mean_displacement_patches', 'mean'),
    p90_displacement_patches=('p90_displacement_patches', 'mean'),
).reset_index()
display(condition_summary_df)


## Cell Role: Visualize Patch Matching Maps

왼쪽부터 clean reference, query image, 같은 좌표 cosine similarity, top-1 cosine similarity, top-1 displacement magnitude를 본다. displacement가 커지면 DINO feature가 같은 물리 위치보다 다른 clean patch를 더 가깝게 본다는 뜻이다.


In [ ]:
import matplotlib.pyplot as plt

VIS_SAMPLE_REFS = sample_refs[:1]
VIS_CONDITIONS = [condition for condition, _, _ in SHIFT_CONDITIONS]


def show_match_row(split: str, image_id: str, condition: str) -> None:
    row = query_df[(query_df['split'].eq(split)) & (query_df['image_id'].eq(image_id)) & (query_df['condition'].eq(condition))].iloc[0]
    result = match_results[(split, image_id, condition)]
    reference_image = np.asarray(Image.open(row['reference_path']).convert('RGB'))
    query_image = np.asarray(Image.open(row['query_path']).convert('RGB'))
    fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
    panels = [
        (reference_image, f'clean reference\n{split}/{image_id}', None),
        (query_image, f'query\n{condition}', None),
        (result.identity_similarity_map, 'same-location cosine', 'viridis'),
        (result.top1_similarity_map, 'top-1 cosine', 'viridis'),
        (result.displacement_norm_map, 'top-1 displacement\n(patches)', 'magma'),
    ]
    for ax, (image, title, cmap) in zip(axes, panels, strict=True):
        im = ax.imshow(image, cmap=cmap, vmin=0 if cmap else None)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
        if cmap:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    metrics = result.metrics
    fig.suptitle(
        ' | '.join([
            f"identity_cos={metrics['identity_similarity_mean']:.3f}",
            f"top1_cos={metrics['top1_similarity_mean']:.3f}",
            f"same_pos={metrics['identity_top1_ratio']:.2f}",
            f"disp={metrics['mean_displacement_patches']:.2f}",
        ]),
        y=1.02,
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()


for split, image_id in VIS_SAMPLE_REFS:
    for condition in VIS_CONDITIONS:
        show_match_row(split, image_id, condition)


## Cell Role: Representation Shift Diagnostics

같은 위치 patch 기준으로 `delta = shifted_feature - clean_feature`를 만들고, feature displacement가 전역적으로 비슷한 방향인지 또는 low-dimensional subspace에 놓이는지 본다. PCA는 두 가지를 본다.

- combined PCA: clean/shifted patch feature를 같은 PCA 공간에 놓고 clean→shift arrow를 그림
- delta PCA: `delta` vector 자체의 분포와 explained variance를 봄


In [ ]:
shift_results = {}
shift_rows = []

for _, row in query_df.iterrows():
    reference_path = Path(row['reference_path'])
    query_path = Path(row['query_path'])
    reference_feature_map = cached_feature_map(reference_path)
    query_feature_map = cached_feature_map(query_path)
    result = analyze_dino_representation_shift(reference_feature_map, query_feature_map)
    key = (row['split'], row['image_id'], row['condition'])
    shift_results[key] = result
    summary = summarize_dino_representation_shift(result)
    summary.update(row.to_dict())
    shift_rows.append(summary)

shift_df = pd.DataFrame(shift_rows)
SHIFT_SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_dino_representation_shift_summary.csv'
shift_df.to_csv(SHIFT_SUMMARY_CSV, index=False)
print(f'saved representation shift summary: {SHIFT_SUMMARY_CSV}')
display(shift_df)

shift_split_condition_summary_df = shift_df.groupby(['split', 'condition']).agg(
    n=('image_id', 'count'),
    mean_delta_norm=('mean_delta_norm', 'mean'),
    mean_delta_vector_norm=('mean_delta_vector_norm', 'mean'),
    direction_consistency=('delta_direction_consistency_mean', 'mean'),
    pairwise_delta_cosine=('pairwise_delta_cosine_mean', 'mean'),
    residual_after_mean_shift=('residual_after_mean_shift_mean', 'mean'),
    delta_pc1_explained=('delta_pc1_explained', 'mean'),
    delta_pc3_cumulative=('delta_pc3_cumulative', 'mean'),
).reset_index()
display(shift_split_condition_summary_df)

shift_condition_summary_df = shift_df.groupby('condition').agg(
    n=('image_id', 'count'),
    mean_delta_norm=('mean_delta_norm', 'mean'),
    mean_delta_vector_norm=('mean_delta_vector_norm', 'mean'),
    direction_consistency=('delta_direction_consistency_mean', 'mean'),
    pairwise_delta_cosine=('pairwise_delta_cosine_mean', 'mean'),
    residual_after_mean_shift=('residual_after_mean_shift_mean', 'mean'),
    delta_pc1_explained=('delta_pc1_explained', 'mean'),
    delta_pc3_cumulative=('delta_pc3_cumulative', 'mean'),
).reset_index()
display(shift_condition_summary_df)


## Cell Role: PCA Arrow Plot

DINO feature를 PCA 2D에 투영한 뒤, 같은 patch의 clean 위치에서 shifted 위치로 arrow를 그린다. 전역 nuisance direction이 강하면 arrow가 비슷한 방향으로 정렬된다. patch별 반응이 다르면 arrow 방향이 흩어진다.


In [ ]:
PCA_SPLIT, PCA_SAMPLE_ID = sample_refs[0]
PCA_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']
PCA_ARROW_STRIDE = 2

fig, axes = plt.subplots(1, len(PCA_CONDITIONS), figsize=(5 * len(PCA_CONDITIONS), 4.5))
if len(PCA_CONDITIONS) == 1:
    axes = [axes]

for ax, condition in zip(axes, PCA_CONDITIONS, strict=True):
    result = shift_results[(PCA_SPLIT, PCA_SAMPLE_ID, condition)]
    clean_xy = result.clean_pca_xy
    query_xy = result.query_pca_xy
    indices = np.arange(len(clean_xy))[::PCA_ARROW_STRIDE]
    ax.scatter(clean_xy[:, 0], clean_xy[:, 1], s=14, alpha=0.45, label='clean')
    ax.scatter(query_xy[:, 0], query_xy[:, 1], s=14, alpha=0.45, label='shifted')
    ax.quiver(
        clean_xy[indices, 0],
        clean_xy[indices, 1],
        query_xy[indices, 0] - clean_xy[indices, 0],
        query_xy[indices, 1] - clean_xy[indices, 1],
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.004,
        alpha=0.65,
    )
    ax.set_title(
        f"{condition}"
        f"dir={result.metrics['delta_direction_consistency_mean']:.2f} "
        f"pc1={result.metrics['delta_pc1_explained']:.2f}"
    )
    ax.set_xlabel('PCA-1')
    ax.set_ylabel('PCA-2')
    ax.grid(True, alpha=0.2)
axes[0].legend(loc='best')
plt.tight_layout()
plt.show()


## Cell Role: Delta PCA And Delta Maps

`delta` vector만 PCA에 올려 condition별 분포를 본다. 오른쪽 map들은 patch별 delta norm과 평균 delta 방향과의 cosine similarity다.


In [ ]:
DELTA_SPLIT, DELTA_SAMPLE_ID = sample_refs[0]
DELTA_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']

fig, axes = plt.subplots(1, len(DELTA_CONDITIONS), figsize=(4.5 * len(DELTA_CONDITIONS), 4))
if len(DELTA_CONDITIONS) == 1:
    axes = [axes]
for ax, condition in zip(axes, DELTA_CONDITIONS, strict=True):
    result = shift_results[(DELTA_SPLIT, DELTA_SAMPLE_ID, condition)]
    xy = result.delta_pca_xy
    sc = ax.scatter(xy[:, 0], xy[:, 1], c=result.delta_norm_map.reshape(-1), cmap='magma', s=28)
    ax.set_title(
        f"{condition}"
        f"PC1={result.metrics['delta_pc1_explained']:.2f}, "
        f"PC1-3={result.metrics['delta_pc3_cumulative']:.2f}"
    )
    ax.set_xlabel('delta PCA-1')
    ax.set_ylabel('delta PCA-2')
    ax.grid(True, alpha=0.2)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

for condition in DELTA_CONDITIONS:
    result = shift_results[(DELTA_SPLIT, DELTA_SAMPLE_ID, condition)]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    im0 = axes[0].imshow(result.delta_norm_map, cmap='magma')
    axes[0].set_title(f'{condition}delta norm')
    axes[0].axis('off')
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    im1 = axes[1].imshow(result.delta_cosine_to_mean_map, cmap='coolwarm', vmin=-1, vmax=1)
    axes[1].set_title('cosine to mean delta')
    axes[1].axis('off')
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


## Cell Role: Displacement Vector Field

Top-1 clean patch가 query patch 기준 어느 방향으로 이동했는지 quiver로 본다. position shift는 실제 geometry가 바뀌므로 displacement가 커지는 것이 정상이고, brightness/noise/blur에서 displacement가 커지면 DINO feature identity가 흔들린다는 신호다.


In [ ]:
QUIVER_SPLIT, QUIVER_SAMPLE_ID = sample_refs[0]
QUIVER_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']

fig, axes = plt.subplots(1, len(QUIVER_CONDITIONS), figsize=(4.5 * len(QUIVER_CONDITIONS), 4))
if len(QUIVER_CONDITIONS) == 1:
    axes = [axes]

for ax, condition in zip(axes, QUIVER_CONDITIONS, strict=True):
    result = match_results[(QUIVER_SPLIT, QUIVER_SAMPLE_ID, condition)]
    height, width = result.grid_shape
    ys, xs = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
    ax.imshow(result.displacement_norm_map, cmap='magma')
    ax.quiver(
        xs,
        ys,
        result.displacement_x_map,
        result.displacement_y_map,
        color='cyan',
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.004,
    )
    ax.set_title(condition)
    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(height - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


## Cell Role: Position Shift Object-Level Correspondence

Position shift는 같은 grid 좌표 비교가 불리하므로, shifted query patch가 clean image의 어느 patch를 top-1 nearest neighbor로 선택하는지 side-by-side line으로 본다. 선이 같은 물체 영역끼리 연결되면 DINO representation은 geometry shift에도 object-level correspondence를 어느 정도 유지하는 것이다.


In [ ]:
from matplotlib.collections import LineCollection

POSITION_VIS_REFS = sample_refs[:2]
POSITION_VIS_CONDITION = 'position_shift_high'
POSITION_VIS_STRIDE = 2
POSITION_VIS_MIN_SIMILARITY = 0.0  # raise to 0.7~0.8 to show only confident matches
POSITION_VIS_MAX_LINES = 140


def patch_center_xy(grid_x: int, grid_y: int, *, grid_width: int, grid_height: int, image_width: int, image_height: int) -> tuple[float, float]:
    return (
        (grid_x + 0.5) * image_width / grid_width,
        (grid_y + 0.5) * image_height / grid_height,
    )


def show_position_shift_correspondence(split: str, image_id: str) -> None:
    row = query_df[
        query_df['split'].eq(split)
        & query_df['image_id'].eq(image_id)
        & query_df['condition'].eq(POSITION_VIS_CONDITION)
    ].iloc[0]
    result = match_results[(split, image_id, POSITION_VIS_CONDITION)]
    clean_image = np.asarray(Image.open(row['reference_path']).convert('RGB'))
    shifted_image = np.asarray(Image.open(row['query_path']).convert('RGB'))
    image_height, image_width = clean_image.shape[:2]
    gap = max(20, image_width // 12)
    canvas = np.ones((image_height, image_width * 2 + gap, 3), dtype=np.uint8) * 255
    canvas[:, :image_width] = clean_image
    canvas[:, image_width + gap:image_width * 2 + gap] = shifted_image

    grid_height, grid_width = result.grid_shape
    segments = []
    colors = []
    clean_points = []
    shifted_points = []
    for query_y in range(0, grid_height, POSITION_VIS_STRIDE):
        for query_x in range(0, grid_width, POSITION_VIS_STRIDE):
            similarity = float(result.top1_similarity_map[query_y, query_x])
            if similarity < POSITION_VIS_MIN_SIMILARITY:
                continue
            matched_index = int(result.top1_index_map[query_y, query_x])
            ref_y, ref_x = divmod(matched_index, grid_width)
            ref_x_px, ref_y_px = patch_center_xy(
                ref_x,
                ref_y,
                grid_width=grid_width,
                grid_height=grid_height,
                image_width=image_width,
                image_height=image_height,
            )
            query_x_px, query_y_px = patch_center_xy(
                query_x,
                query_y,
                grid_width=grid_width,
                grid_height=grid_height,
                image_width=image_width,
                image_height=image_height,
            )
            query_x_px += image_width + gap
            segments.append([(ref_x_px, ref_y_px), (query_x_px, query_y_px)])
            colors.append(similarity)
            clean_points.append((ref_x_px, ref_y_px))
            shifted_points.append((query_x_px, query_y_px))

    if len(segments) > POSITION_VIS_MAX_LINES:
        order = np.argsort(colors)[-POSITION_VIS_MAX_LINES:]
        segments = [segments[index] for index in order]
        colors = [colors[index] for index in order]
        clean_points = [clean_points[index] for index in order]
        shifted_points = [shifted_points[index] for index in order]

    fig, ax = plt.subplots(1, 1, figsize=(14, 6))
    ax.imshow(canvas)
    line_collection = LineCollection(segments, cmap='viridis', linewidths=0.8, alpha=0.55)
    line_collection.set_array(np.asarray(colors, dtype=np.float32))
    line_collection.set_clim(0.0, 1.0)
    ax.add_collection(line_collection)
    if clean_points:
        clean_xy = np.asarray(clean_points)
        shifted_xy = np.asarray(shifted_points)
        ax.scatter(clean_xy[:, 0], clean_xy[:, 1], s=12, c='white', edgecolors='black', linewidths=0.3, label='matched clean patch')
        ax.scatter(shifted_xy[:, 0], shifted_xy[:, 1], s=12, c='cyan', edgecolors='black', linewidths=0.3, label='query shifted patch')
    ax.axvline(image_width + gap / 2, color='black', linewidth=1)
    ax.text(image_width / 2, 16, f'clean reference: {split}/{image_id}', color='white', ha='center', va='top', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.5, 'pad': 3})
    ax.text(image_width + gap + image_width / 2, 16, 'position shifted query', color='white', ha='center', va='top', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.5, 'pad': 3})
    metrics = result.metrics
    ax.set_title(
        f"object-level top1 patch correspondences | "
        f"top1_cos={metrics['top1_similarity_mean']:.3f}, "
        f"same_pos={metrics['identity_top1_ratio']:.2f}, "
        f"local3x3={metrics['local_3x3_top1_ratio']:.2f}"
    )
    ax.axis('off')
    ax.legend(loc='lower center', ncols=2)
    fig.colorbar(line_collection, ax=ax, fraction=0.025, pad=0.01, label='top-1 cosine')
    plt.tight_layout()
    plt.show()


for split, image_id in POSITION_VIS_REFS:
    show_position_shift_correspondence(split, image_id)


## Result Notes

전체 `train/validation/test good` 기준으로 보면 DINOv2 final patch feature는 brightness/blur/noise에서는 같은 위치 또는 주변 3x3 위치와 대부분 잘 대응된다. 반대로 position shift는 실제 content geometry가 바뀌므로 same-location matching이 깨지는 것이 정상이다.

현재 확인된 패턴은 다음과 같다.

- `brightness_high`: same-location cosine이 매우 높고 top-1도 거의 같은 위치에 머문다.
- `gaussian_blur_high`: cosine은 조금 낮아지지만 대부분 3x3 주변 안에서 대응된다.
- `gaussian_noise_high`: blur보다 더 흔들리지만 local correspondence는 아직 상당 부분 유지된다.
- `position_shift_high`: top-1은 다른 위치로 크게 이동한다. 이 조건은 representation shift라기보다 alignment/geometry 문제로 분리해서 봐야 한다.
- representation delta는 완전한 single global vector는 아니다. 다만 direction consistency와 PCA explained variance가 양수라서 약한 shared nuisance subspace/moment shift 가정은 계속 검토할 가치가 있다.


## Cell Role: Dataset-Level Similarity Gap

전체 데이터에 대해 `gap = top1 cosine - same-location cosine` 분포를 본다. gap이 0에 가까우면 같은 위치 patch가 이미 가장 좋은 대응이고, gap이 커지면 다른 위치의 clean patch가 더 유사하다는 뜻이다.


In [ ]:
gap_rows = []

for _, row in query_df.iterrows():
    key = (row['split'], row['image_id'], row['condition'])
    result = match_results[key]

    identity = result.identity_similarity_map.reshape(-1)
    top1 = result.top1_similarity_map.reshape(-1)
    displacement = result.displacement_norm_map.reshape(-1)
    gap = top1 - identity

    gap_rows.append({
        'split': row['split'],
        'image_id': row['image_id'],
        'condition': row['condition'],
        'gap_mean': float(np.mean(gap)),
        'gap_median': float(np.median(gap)),
        'gap_p90': float(np.quantile(gap, 0.90)),
        'gap_p95': float(np.quantile(gap, 0.95)),
        'gap_max': float(np.max(gap)),
        'same_position_ratio': float(np.mean(displacement == 0)),
        'near_3x3_ratio': float(np.mean(displacement <= np.sqrt(2))),
        'far_match_ratio': float(np.mean(displacement > np.sqrt(2))),
        'identity_similarity_mean': float(np.mean(identity)),
        'top1_similarity_mean': float(np.mean(top1)),
    })

gap_df = pd.DataFrame(gap_rows)
display(gap_df)

gap_split_condition_df = gap_df.groupby(['split', 'condition']).agg(
    n=('image_id', 'count'),
    gap_mean=('gap_mean', 'mean'),
    gap_p90=('gap_p90', 'mean'),
    gap_p95=('gap_p95', 'mean'),
    gap_max=('gap_max', 'mean'),
    same_position_ratio=('same_position_ratio', 'mean'),
    near_3x3_ratio=('near_3x3_ratio', 'mean'),
    far_match_ratio=('far_match_ratio', 'mean'),
    identity_similarity_mean=('identity_similarity_mean', 'mean'),
    top1_similarity_mean=('top1_similarity_mean', 'mean'),
).reset_index()
display(gap_split_condition_df)

gap_condition_df = gap_df.groupby('condition').agg(
    n=('image_id', 'count'),
    gap_mean=('gap_mean', 'mean'),
    gap_p90=('gap_p90', 'mean'),
    gap_p95=('gap_p95', 'mean'),
    gap_max=('gap_max', 'mean'),
    same_position_ratio=('same_position_ratio', 'mean'),
    near_3x3_ratio=('near_3x3_ratio', 'mean'),
    far_match_ratio=('far_match_ratio', 'mean'),
    identity_similarity_mean=('identity_similarity_mean', 'mean'),
    top1_similarity_mean=('top1_similarity_mean', 'mean'),
).reset_index()
display(gap_condition_df)


## Cell Role: Dataset-Level Histograms And Boxplots

전체 이미지 기준으로 condition별 matching score, top-1 gap, displacement 분포를 직관적으로 본다.


In [ ]:
conditions = [condition for condition, _, _ in SHIFT_CONDITIONS if condition != 'clean_identity']
plot_df = summary_df[summary_df['condition'].ne('clean_identity')].copy()

metrics = [
    ('identity_similarity_mean', 'same-location cosine'),
    ('top1_similarity_mean', 'top-1 cosine'),
    ('identity_top1_ratio', 'same-position top1 ratio'),
    ('local_3x3_top1_ratio', 'local 3x3 top1 ratio'),
    ('mean_displacement_patches', 'mean top1 displacement'),
    ('p90_displacement_patches', 'p90 top1 displacement'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.reshape(-1)
for ax, (metric, title) in zip(axes, metrics, strict=True):
    plot_df.boxplot(column=metric, by='condition', ax=ax, rot=30)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.2)
fig.suptitle('DINO patch matching metrics across all clean good images')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(conditions), figsize=(4.5 * len(conditions), 3.5), sharey=True)
if len(conditions) == 1:
    axes = [axes]
for ax, condition in zip(axes, conditions, strict=True):
    values = []
    for _, row in query_df[query_df['condition'].eq(condition)].iterrows():
        key = (row['split'], row['image_id'], row['condition'])
        result = match_results[key]
        gap = result.top1_similarity_map.reshape(-1) - result.identity_similarity_map.reshape(-1)
        values.append(gap)
    values = np.concatenate(values) if values else np.array([])
    ax.hist(values, bins=60, alpha=0.85)
    ax.axvline(0.0, color='black', linestyle='--', linewidth=1)
    ax.axvline(np.quantile(values, 0.90), color='red', linestyle='--', linewidth=1)
    ax.set_title(f'{condition}\nmean={np.mean(values):.4f}, p90={np.quantile(values, 0.90):.4f}')
    ax.set_xlabel('top1 cosine - same-location cosine')
    ax.grid(True, alpha=0.2)
axes[0].set_ylabel('patch count')
plt.tight_layout()
plt.show()


## Cell Role: Dataset-Level Heatmaps

개별 샘플 quiver 대신 전체 데이터에서 평균 displacement, 평균 vector field, 평균 representation delta map을 본다.


In [ ]:
fig, axes = plt.subplots(1, len(conditions), figsize=(4.2 * len(conditions), 4))
if len(conditions) == 1:
    axes = [axes]
for ax, condition in zip(axes, conditions, strict=True):
    maps = [
        match_results[(row['split'], row['image_id'], row['condition'])].displacement_norm_map
        for _, row in query_df[query_df['condition'].eq(condition)].iterrows()
    ]
    mean_map = np.mean(np.stack(maps, axis=0), axis=0)
    im = ax.imshow(mean_map, cmap='magma')
    ax.set_title(f'{condition}\nmean={mean_map.mean():.2f}, max={mean_map.max():.2f}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(conditions), figsize=(4.2 * len(conditions), 4))
if len(conditions) == 1:
    axes = [axes]
for ax, condition in zip(axes, conditions, strict=True):
    dx_maps = []
    dy_maps = []
    for _, row in query_df[query_df['condition'].eq(condition)].iterrows():
        result = match_results[(row['split'], row['image_id'], row['condition'])]
        dx_maps.append(result.displacement_x_map)
        dy_maps.append(result.displacement_y_map)
    mean_dx = np.mean(np.stack(dx_maps, axis=0), axis=0)
    mean_dy = np.mean(np.stack(dy_maps, axis=0), axis=0)
    mean_norm = np.sqrt(mean_dx**2 + mean_dy**2)
    h, w = mean_dx.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    ax.imshow(mean_norm, cmap='magma')
    ax.quiver(xs, ys, mean_dx, mean_dy, color='cyan', angles='xy', scale_units='xy', scale=1, width=0.004)
    ax.set_title(f'{condition}\nmean vector norm={mean_norm.mean():.2f}')
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, len(conditions), figsize=(4.2 * len(conditions), 7.5))
if len(conditions) == 1:
    axes = np.asarray(axes).reshape(2, 1)
for col, condition in enumerate(conditions):
    delta_norm_maps = []
    cosine_maps = []
    for _, row in query_df[query_df['condition'].eq(condition)].iterrows():
        result = shift_results[(row['split'], row['image_id'], row['condition'])]
        delta_norm_maps.append(result.delta_norm_map)
        cosine_maps.append(result.delta_cosine_to_mean_map)
    mean_delta_norm = np.mean(np.stack(delta_norm_maps, axis=0), axis=0)
    mean_cosine = np.mean(np.stack(cosine_maps, axis=0), axis=0)
    im0 = axes[0, col].imshow(mean_delta_norm, cmap='magma')
    axes[0, col].set_title(f'{condition}\nmean delta norm')
    axes[0, col].axis('off')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)
    im1 = axes[1, col].imshow(mean_cosine, cmap='coolwarm', vmin=-1, vmax=1)
    axes[1, col].set_title('cosine to mean delta')
    axes[1, col].axis('off')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## Cell Role: Split-Level Heatmaps

train/validation/test good 사이에 DINO shift response가 크게 다른지 확인한다.


In [ ]:
metric_names = [
    'identity_similarity_mean',
    'identity_top1_ratio',
    'mean_displacement_patches',
]

fig, axes = plt.subplots(1, len(metric_names), figsize=(5 * len(metric_names), 4))
if len(metric_names) == 1:
    axes = [axes]
for ax, metric in zip(axes, metric_names, strict=True):
    pivot = summary_df.pivot_table(index='split', columns='condition', values=metric, aggfunc='mean')
    im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
    ax.set_title(metric)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha='right')
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f'{pivot.values[i, j]:.2f}', ha='center', va='center', color='white')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## Cell Role: DINO Layer Feature Ablation

DINOv2 final block만 쓰는 대신 intermediate block feature도 같은 방식으로 비교한다. 이 ablation은 mask/prototype/OT 없이 feature layer만 바꿔서 patch correspondence와 representation shift가 어느 층에서 더 안정적인지 본다.

`get_intermediate_layers`를 사용해 block 3/6/9/12 feature를 batch로 한 번에 추출한다. `block_12`는 final-layer patch token에 가까운 비교군이다.


In [ ]:
DINO_LAYER_SPECS = [
    ('block_03', 2),
    ('block_06', 5),
    ('block_09', 8),
    ('block_12', 11),
]
LAYER_ABLATION_CSV = PROBE_ROOT / f'{CATEGORY}_dino_layer_ablation_summary.csv'


@torch.no_grad()
def extract_dino_layer_feature_maps_batch(
    image_paths: list[Path],
    *,
    layer_specs: list[tuple[str, int]] = DINO_LAYER_SPECS,
    batch_size: int = DINO_BATCH_SIZE,
) -> dict[str, dict[str, np.ndarray]]:
    outputs = {name: {} for name, _ in layer_specs}
    unique_paths = list(dict.fromkeys(Path(path) for path in image_paths))
    layer_indices = [index for _, index in layer_specs]
    for start in range(0, len(unique_paths), batch_size):
        batch_paths = unique_paths[start:start + batch_size]
        batch = preprocess_dino_batch(batch_paths)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_DINO_AMP):
            layer_outputs = dino_model.get_intermediate_layers(
                batch,
                n=layer_indices,
                reshape=False,
                return_class_token=False,
                norm=True,
            )
        for (layer_name, _), tokens in zip(layer_specs, layer_outputs, strict=True):
            tokens = tokens.detach().cpu().float()
            grid = int(math.sqrt(tokens.shape[1]))
            if grid * grid != tokens.shape[1]:
                raise ValueError(f'DINO token count is not square for {layer_name}: {tokens.shape[1]}')
            feature_maps = tokens.reshape(tokens.shape[0], grid, grid, tokens.shape[-1]).numpy()
            for path, feature_map in zip(batch_paths, feature_maps, strict=True):
                outputs[layer_name][str(path)] = feature_map
        print(f'DINO layer batch {min(start + batch_size, len(unique_paths))}/{len(unique_paths)}')
    return outputs


layer_feature_cache = extract_dino_layer_feature_maps_batch(unique_feature_paths, batch_size=DINO_BATCH_SIZE)

layer_rows = []
for feature_source, path_to_features in layer_feature_cache.items():
    for _, row in query_df.iterrows():
        reference_path = str(Path(row['reference_path']))
        query_path = str(Path(row['query_path']))
        reference_feature_map = path_to_features[reference_path]
        query_feature_map = path_to_features[query_path]
        match_result = match_dino_patch_grid(reference_feature_map, query_feature_map)
        shift_result = analyze_dino_representation_shift(reference_feature_map, query_feature_map)
        record = summarize_dino_patch_match(match_result)
        record.update({f'shift_{key}': value for key, value in summarize_dino_representation_shift(shift_result).items()})
        record.update({
            'feature_source': feature_source,
            'split': row['split'],
            'image_id': row['image_id'],
            'condition': row['condition'],
            'augmentation_type': row['augmentation_type'],
            'severity': row['severity'],
        })
        layer_rows.append(record)

layer_ablation_df = pd.DataFrame(layer_rows)
layer_ablation_df.to_csv(LAYER_ABLATION_CSV, index=False)
print(f'saved layer ablation summary: {LAYER_ABLATION_CSV}')
display(layer_ablation_df)

layer_condition_summary_df = layer_ablation_df.groupby(['feature_source', 'condition']).agg(
    n=('image_id', 'count'),
    identity_similarity_mean=('identity_similarity_mean', 'mean'),
    top1_similarity_mean=('top1_similarity_mean', 'mean'),
    identity_top1_ratio=('identity_top1_ratio', 'mean'),
    local_3x3_top1_ratio=('local_3x3_top1_ratio', 'mean'),
    mean_displacement_patches=('mean_displacement_patches', 'mean'),
    shift_mean_delta_norm=('shift_mean_delta_norm', 'mean'),
    shift_direction_consistency=('shift_delta_direction_consistency_mean', 'mean'),
    shift_delta_pc3_cumulative=('shift_delta_pc3_cumulative', 'mean'),
).reset_index()
display(layer_condition_summary_df)


## Cell Role: Layer Ablation Plots

각 DINO block feature가 condition shift에서 얼마나 위치 correspondence를 유지하는지 비교한다.


In [ ]:
layer_plot_df = layer_ablation_df[layer_ablation_df['condition'].ne('clean_identity')].copy()
layer_metrics = [
    ('identity_similarity_mean', 'same-location cosine'),
    ('identity_top1_ratio', 'same-position top1 ratio'),
    ('local_3x3_top1_ratio', 'local 3x3 top1 ratio'),
    ('mean_displacement_patches', 'mean top1 displacement'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.reshape(-1)
for ax, (metric, title) in zip(axes, layer_metrics, strict=True):
    for condition in conditions:
        condition_df = layer_condition_summary_df[layer_condition_summary_df['condition'].eq(condition)]
        ax.plot(condition_df['feature_source'], condition_df[metric], marker='o', label=condition)
    ax.set_title(title)
    ax.set_xlabel('DINO feature source')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.2)
axes[0].legend(loc='best')
plt.tight_layout()
plt.show()

shift_layer_metrics = [
    ('shift_mean_delta_norm', 'mean delta norm'),
    ('shift_direction_consistency', 'direction consistency'),
    ('shift_delta_pc3_cumulative', 'delta PC1-3 cumulative'),
]
fig, axes = plt.subplots(1, len(shift_layer_metrics), figsize=(5 * len(shift_layer_metrics), 4))
if len(shift_layer_metrics) == 1:
    axes = [axes]
for ax, (metric, title) in zip(axes, shift_layer_metrics, strict=True):
    for condition in conditions:
        condition_df = layer_condition_summary_df[layer_condition_summary_df['condition'].eq(condition)]
        ax.plot(condition_df['feature_source'], condition_df[metric], marker='o', label=condition)
    ax.set_title(title)
    ax.set_xlabel('DINO feature source')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.2)
axes[0].legend(loc='best')
plt.tight_layout()
plt.show()
